# Mesh Tutorial 2: Operations and Transformations

This tutorial covers mesh manipulation operations in PhysicsNeMo-Mesh:

1. **Geometric Transformations**: translate, rotate, scale, arbitrary linear transforms
2. **Subdivision**: Refine meshes with different smoothing schemes
3. **Slicing**: Extract subsets of points or cells
4. **Merging**: Combine multiple meshes into one
5. **Boundary & Facet Extraction**: Get boundaries and lower-dimensional elements
6. **Data Conversion**: Move data between points and cells
7. **Topology Checks**: Watertight and manifold detection

In [ ]:
import torch
import math

from physicsnemo.mesh import Mesh
from physicsnemo.mesh.primitives.surfaces import sphere_icosahedral, torus
from physicsnemo.mesh.primitives.volumes import cube_volume
from physicsnemo.mesh.primitives.planar import unit_square

## Section 1: Geometric Transformations

PhysicsNeMo-Mesh provides standard geometric transformations that operate on the mesh geometry.
All transformations return a **new mesh** (they don't modify in place).

### Translation

Move all points by a fixed offset vector.

In [ ]:
sphere = sphere_icosahedral.load(subdivisions=2)

# Translate by a vector
translated = sphere.translate([5.0, 0.0, 0.0])

print(f"Original center: {sphere.points.mean(dim=0)}")
print(f"Translated center: {translated.points.mean(dim=0)}")

### Scaling

Scale the mesh uniformly or anisotropically (different factors per axis).

In [ ]:
sphere = sphere_icosahedral.load(subdivisions=2)

# Uniform scaling: double the size
scaled_uniform = sphere.scale(2.0)
print(f"Original extent: {sphere.points.max(dim=0).values - sphere.points.min(dim=0).values}")
print(f"Uniform 2x: {scaled_uniform.points.max(dim=0).values - scaled_uniform.points.min(dim=0).values}")

# Anisotropic scaling: stretch into an ellipsoid
scaled_aniso = sphere.scale([2.0, 1.0, 0.5])
print(f"Anisotropic: {scaled_aniso.points.max(dim=0).values - scaled_aniso.points.min(dim=0).values}")

# Visualize the ellipsoid
scaled_aniso.draw()

### Rotation

Rotate around an axis by a specified angle (in radians).

- For **2D meshes**: No axis needed (rotation is in the plane)
- For **3D meshes**: Specify the rotation axis

In [ ]:
# Load the bunny for a more interesting example
bunny = torch.load("assets/bunny.pt", weights_only=False).subdivide(1, "loop")

# Rotate 45 degrees around the Z-axis
rotated_z = bunny.rotate(angle=math.pi / 4, axis=[0, 0, 1])

# Rotate 90 degrees around the Y-axis
rotated_y = bunny.rotate(angle=math.pi / 2, axis=[0, 1, 0])

# Rotation around an arbitrary axis
rotated_arbitrary = bunny.rotate(angle=math.pi / 3, axis=[1, 1, 1])

rotated_z.draw()

### Arbitrary Linear Transform

Apply any linear transformation via a matrix. This is the most general transformation,
encompassing rotation, scaling, shearing, and even projection to different dimensions.

In [ ]:
sphere = sphere_icosahedral.load(subdivisions=2)

# Shear transformation
shear_matrix = torch.tensor([
    [1.0, 0.5, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])
sheared = sphere.transform(shear_matrix)
sheared.draw()

In [ ]:
# Projection to 2D (drop the z coordinate)
projection_matrix = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
])
projected = sphere.transform(projection_matrix)
print(f"Original: {sphere.n_spatial_dims}D")
print(f"Projected: {projected.n_spatial_dims}D")
projected.draw()

## Section 2: Subdivision

Subdivision refines a mesh by splitting each cell into smaller cells. This is useful for:
- Increasing mesh resolution
- Smoothing coarse meshes
- Creating smooth surfaces from control meshes

PhysicsNeMo-Mesh supports three subdivision schemes:

| Scheme | Type | Properties |
|--------|------|------------|
| `linear` | Interpolating | Midpoint subdivision, preserves original vertices |
| `loop` | Approximating | C² smooth, moves original vertices |
| `butterfly` | Interpolating | Smooth, preserves original vertices |

In [ ]:
# Start with a coarse icosahedron (20 triangles)
coarse = sphere_icosahedral.load(subdivisions=0)
print(f"Coarse: {coarse.n_points} points, {coarse.n_cells} cells")

# Each level of subdivision multiplies cells by 4 (for triangles)
linear_1 = coarse.subdivide(levels=1, filter="linear")
linear_2 = coarse.subdivide(levels=2, filter="linear")
linear_3 = coarse.subdivide(levels=3, filter="linear")

print(f"Linear 1 level: {linear_1.n_points} points, {linear_1.n_cells} cells")
print(f"Linear 2 levels: {linear_2.n_points} points, {linear_2.n_cells} cells")
print(f"Linear 3 levels: {linear_3.n_points} points, {linear_3.n_cells} cells")

In [ ]:
# Compare subdivision schemes on a coarse mesh
coarse = sphere_icosahedral.load(subdivisions=0)

# Linear: just splits cells, doesn't smooth
linear = coarse.subdivide(levels=2, filter="linear")

# Loop: C² smooth, approximating (moves original vertices)
loop = coarse.subdivide(levels=2, filter="loop")

# Butterfly: smooth, interpolating (preserves original vertices)
butterfly = coarse.subdivide(levels=2, filter="butterfly")

print("Linear subdivision (faceted):")
linear.draw(show_edges=True)

In [ ]:
print("Loop subdivision (smooth, C²):")
loop.draw(show_edges=True)

In [ ]:
print("Butterfly subdivision (smooth, interpolating):")
butterfly.draw(show_edges=True)

### Data Interpolation During Subdivision

When you subdivide a mesh with attached data, the data is automatically interpolated
to the new vertices and cells.

In [ ]:
# Create a mesh with data
mesh = sphere_icosahedral.load(subdivisions=1)

# Add a scalar field based on z-coordinate
mesh.point_data["height"] = mesh.points[:, 2]
print(f"Before: {mesh.n_points} points")

# Subdivide - data is interpolated automatically
refined = mesh.subdivide(levels=2, filter="loop")
print(f"After: {refined.n_points} points")
print(f"Data keys preserved: {list(refined.point_data.keys())}")

refined.draw(point_scalars="height", cmap="viridis", show_edges=False)

## Section 3: Slicing

Slicing extracts a subset of points or cells from a mesh. You can slice by:
- Integer indices
- Boolean masks
- Index arrays

### Slicing Cells

`slice_cells()` keeps only the specified cells. Points are preserved (even unused ones).

In [ ]:
sphere = sphere_icosahedral.load(subdivisions=2)
print(f"Original: {sphere.n_cells} cells")

# Slice using a boolean mask: keep cells with positive x-centroid
mask = sphere.cell_centroids[:, 0] > 0
hemisphere_x = sphere.slice_cells(mask)
print(f"X > 0: {hemisphere_x.n_cells} cells")

hemisphere_x.draw()

In [ ]:
# Slice with compound conditions
mask = (sphere.cell_centroids[:, 0] > 0) & (sphere.cell_centroids[:, 2] > 0)
quadrant = sphere.slice_cells(mask)
print(f"X > 0 and Z > 0: {quadrant.n_cells} cells")

quadrant.draw()

In [ ]:
# Slice by index array
indices = torch.arange(0, sphere.n_cells, 2)  # Every other cell
every_other = sphere.slice_cells(indices)
print(f"Every other cell: {every_other.n_cells} cells")

### Slicing Points

`slice_points()` keeps only the specified points. Cells that reference removed points
are automatically removed, and remaining cell indices are remapped.

In [ ]:
sphere = sphere_icosahedral.load(subdivisions=2)
print(f"Original: {sphere.n_points} points, {sphere.n_cells} cells")

# Keep only points with z > 0
mask = sphere.points[:, 2] > 0
top_half = sphere.slice_points(mask)
print(f"Z > 0: {top_half.n_points} points, {top_half.n_cells} cells")

# Note: cells that cross z=0 are removed (they reference deleted points)
top_half.draw()

## Section 4: Merging Meshes

`Mesh.merge()` combines multiple meshes into a single mesh. The meshes must have:
- Same spatial dimension
- Same manifold dimension
- Same cell_data keys (if any)

In [ ]:
# Create three spheres at different positions
sphere1 = sphere_icosahedral.load(subdivisions=2).translate([-2.0, 0.0, 0.0])
sphere2 = sphere_icosahedral.load(subdivisions=2).translate([0.0, 0.0, 0.0])
sphere3 = sphere_icosahedral.load(subdivisions=2).translate([2.0, 0.0, 0.0])

print(f"Sphere 1: {sphere1.n_points} points, {sphere1.n_cells} cells")
print(f"Sphere 2: {sphere2.n_points} points, {sphere2.n_cells} cells")
print(f"Sphere 3: {sphere3.n_points} points, {sphere3.n_cells} cells")

# Merge them
merged = Mesh.merge([sphere1, sphere2, sphere3])
print(f"\nMerged: {merged.n_points} points, {merged.n_cells} cells")

merged.draw()

In [ ]:
# Merge preserves attached data
sphere1.point_data["id"] = torch.full((sphere1.n_points,), 0.0)
sphere2.point_data["id"] = torch.full((sphere2.n_points,), 1.0)
sphere3.point_data["id"] = torch.full((sphere3.n_points,), 2.0)

merged = Mesh.merge([sphere1, sphere2, sphere3])
merged.draw(point_scalars="id", cmap="Set1", show_edges=False)

## Section 5: Boundary and Facet Extraction

PhysicsNeMo-Mesh can extract:
- **Boundary mesh**: Only the facets that are on the boundary (shared by exactly 1 cell)
- **Facet mesh**: All (n-k)-dimensional facets of an n-dimensional mesh

### Boundary Extraction

Extract the boundary surface of a volume mesh.

In [ ]:
# Load a tetrahedral volume mesh
cube = cube_volume.load(n=4)
print(f"Volume mesh: {cube}")
print(f"  Manifold dim: {cube.n_manifold_dims} (tetrahedra)")

# Extract the boundary surface
boundary = cube.get_boundary_mesh()
print(f"\nBoundary mesh: {boundary}")
print(f"  Manifold dim: {boundary.n_manifold_dims} (triangles)")

boundary.draw()

In [ ]:
# For a closed surface mesh, the boundary is empty
sphere = sphere_icosahedral.load(subdivisions=2)
sphere_boundary = sphere.get_boundary_mesh()
print(f"Sphere boundary: {sphere_boundary.n_cells} cells (should be 0 for closed surface)")

### Facet Extraction

Extract ALL lower-dimensional elements (not just boundary).

In [ ]:
# Extract all edges from a triangle mesh
sphere = sphere_icosahedral.load(subdivisions=1)
print(f"Triangle mesh: {sphere}")

# Get codimension-1 facets: triangles -> edges
edges = sphere.get_facet_mesh(manifold_codimension=1)
print(f"\nEdge mesh: {edges}")
print(f"  Each edge is shared by 2 triangles (interior) or 1 triangle (boundary)")

edges.draw()

In [ ]:
# Extract all faces from a tetrahedral mesh
cube = cube_volume.load(n=3)
print(f"Tet mesh: {cube}")

# Codimension-1: tetrahedra -> triangular faces
all_faces = cube.get_facet_mesh(manifold_codimension=1)
print(f"All triangular faces: {all_faces}")

# Codimension-2: tetrahedra -> edges
all_edges = cube.get_facet_mesh(manifold_codimension=2)
print(f"All edges: {all_edges}")

## Section 6: Data Conversion

Sometimes you need to move data between points and cells:
- **cell_data_to_point_data**: Average cell values to vertices
- **point_data_to_cell_data**: Average vertex values to cells

In [ ]:
mesh = sphere_icosahedral.load(subdivisions=2)

# Create cell data
mesh.cell_data["cell_value"] = torch.randn(mesh.n_cells)
print(f"Before: point_data keys = {list(mesh.point_data.keys())}")

# Convert to point data (averages from adjacent cells)
mesh_with_point_data = mesh.cell_data_to_point_data()
print(f"After: point_data keys = {list(mesh_with_point_data.point_data.keys())}")

In [ ]:
# Convert point data to cell data
mesh = sphere_icosahedral.load(subdivisions=2)
mesh.point_data["temperature"] = mesh.points[:, 2]  # z-coordinate as temperature

print(f"Before: cell_data keys = {list(mesh.cell_data.keys())}")

mesh_with_cell_data = mesh.point_data_to_cell_data()
print(f"After: cell_data keys = {list(mesh_with_cell_data.cell_data.keys())}")

mesh_with_cell_data.draw(cell_scalars="temperature", cmap="coolwarm")

## Section 7: Topology Checks

PhysicsNeMo-Mesh can check topological properties of meshes.

### Watertight Check

A mesh is **watertight** (or "closed") if it has no boundary - every facet is shared
by exactly 2 cells.

In [ ]:
# Closed sphere - watertight
sphere = sphere_icosahedral.load(subdivisions=2)
print(f"Sphere is watertight: {sphere.is_watertight()}")

# Hemisphere - not watertight (has boundary)
hemisphere = sphere.slice_cells(sphere.cell_centroids[:, 2] > 0)
print(f"Hemisphere is watertight: {hemisphere.is_watertight()}")

### Manifold Check

A mesh is a **manifold** if it locally looks like Euclidean space at every point.
Non-manifold meshes have edges shared by more than 2 faces or "pinched" vertices.

In [ ]:
# Valid manifold
sphere = sphere_icosahedral.load(subdivisions=2)
print(f"Sphere is manifold: {sphere.is_manifold()}")

# Also valid manifold (with boundary)
hemisphere = sphere.slice_cells(sphere.cell_centroids[:, 2] > 0)
print(f"Hemisphere is manifold: {hemisphere.is_manifold()}")

## Summary

In this tutorial, you learned how to manipulate meshes:

1. **Transformations**: `translate()`, `rotate()`, `scale()`, `transform()`
2. **Subdivision**: `subdivide(levels, filter)` with linear/loop/butterfly schemes
3. **Slicing**: `slice_cells()` and `slice_points()` with masks or indices
4. **Merging**: `Mesh.merge([mesh1, mesh2, ...])`
5. **Boundaries**: `get_boundary_mesh()` and `get_facet_mesh()`
6. **Data conversion**: `cell_data_to_point_data()` and `point_data_to_cell_data()`
7. **Topology**: `is_watertight()` and `is_manifold()`

---

### Next Steps

- **Tutorial 3: Discrete Calculus** - Compute gradients, divergence, curl, and curvature
- **Tutorial 4: Neighbors & Spatial Queries** - Adjacency, BVH, sampling
- **Tutorial 5: Quality & Repair** - Mesh validation and repair